# Proyecto final

**Eliana Del Río**

**Cuatro pasos.**

La estructura que tiene que tener el notebook para el proyecto final:

| Paso | Qué hacer |
|---|---|
| **Bajar** | Conseguir los datos crudos (API, scraping, archivo) |
| **Limpiar** | Quedarse con los campos que les importan, en el tipo correcto |
| **Procesar** | Hacer algo con esos datos (agregar, calcular, transformar) |
| **Guardar** | Dejar una tabla limpia en CSV |

Para hacerlo tangible, hago un ejemplo con un caso solo de mi proyecto.


## 1 · Bajar

In [2]:
# Bajar la información del html con extracción de título y descripción 
# Código con IA

import pandas as pd
import requests
from bs4 import BeautifulSoup
import json

# Definir la URL del artículo de 'la diaria'
url = "https://ladiaria.com.uy/ambiente/articulo/2026/2/un-problema-que-se-viene-agravando-desde-hace-muchos-anos-organizacion-exige-acciones-para-proteger-cuenca-del-rio-santa-lucia/"

# Nota: Petición a la página para que el diario no bloquee script, pasar un 'User-Agent' (simula un navegador)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print("Descargando la página...")
respuesta = requests.get(url, headers=headers)

if respuesta.status_code == 200:
    soup = BeautifulSoup(respuesta.text, 'html.parser')
    print("[OK] HTML descargado y guardado en la variable 'soup'")
else:
    print(f"[ERROR] Código de estado: {respuesta.status_code}")

Descargando la página...
[OK] HTML descargado y guardado en la variable 'soup'


### Extracción de información: 

In [3]:
# A) Extraer Título
titulo_nodo = soup.find('h1')
titulo = titulo_nodo.text.strip() if titulo_nodo else "No encontrado"

# B) Extraer Copete / Resumen
copete_nodo = soup.find('meta', attrs={'name': 'description'})
copete = copete_nodo.get('content') if copete_nodo else "No encontrado"

# C) Extraer Fecha de publicación (usando los metadatos estructurados JSON de la página)
fecha = "No encontrada"
script_datos = soup.find('script', type='application/ld+json')
if script_datos:
    try:
        datos_json = json.loads(script_datos.string)
        if 'datePublished' in datos_json:
            fecha = datos_json['datePublished'][:10] # Formato AAAA-MM-DD
    except Exception:
        pass

# D) Extraer Cuerpo del texto
# Agarrar los párrafos largos (> 30 caracteres) para esquivar los textos del menú
parrafos = soup.find_all('p')
texto_cuerpo = " ".join([p.text.strip() for p in parrafos if len(p.text.strip()) > 30])

# --- Verificar visualmente qué se extrajo ---
print("--- DATOS EXTRAÍDOS ---")
print("FECHA:", fecha)
print("TÍTULO:", titulo)
print("COPETE:", copete[:100] + "...") # Mostrar solo el principio
print("CUERPO (Primeros 200 caracteres):", texto_cuerpo[:200] + "...")

--- DATOS EXTRAÍDOS ---
FECHA: 2026-02-24
TÍTULO: “Un problema que se viene agravando desde hace muchos años”: organización exige acciones para proteger cuenca del río Santa Lucía
COPETE: La Asamblea por el Agua del Río Santa Lucía denuncia “falta de acciones y de información confiable” ...
CUERPO (Primeros 200 caracteres): Aguas Corrientes, en las costas del río Santa Lucía sobre la ruta 36, en Canelones (archivo, enero de 2025). La Asamblea por el Agua del Río Santa Lucía denuncia “falta de acciones y de información co...


### Estructurar información en una DataFrame

In [5]:
# Crear un diccionario con la estructura de tu matriz de conflictos
datos_conflicto = {
    "id_conflicto": [1],
    "medio_prensa": ["la diaria"],
    "url": [url],
    "fecha_publicada": [fecha],
    "titulo": [titulo],
    "copete": [copete],
    "texto_completo": [texto_cuerpo],
    "tipo_problema": ["Por definir en etapa de análisis"],
    "actores_organizaciones": ["Por definir en etapa de análisis"]
}

# Convertirel diccionario en un DataFrame de Pandas
df_proyectos = pd.DataFrame(datos_conflicto)

# Visualizar cómo quedó armada la tabla 
df_proyectos

,id_conflicto,medio_prensa,url,fecha_publicada,titulo,copete,texto_completo,tipo_problema,actores_organizaciones
0,1,la diaria,https://ladiaria.com.uy/ambiente/articulo/2026...,2026-02-24,“Un problema que se viene agravando desde hace...,La Asamblea por el Agua del Río Santa Lucía de...,"Aguas Corrientes, en las costas del río Santa ...",Por definir en etapa de análisis,Por definir en etapa de análisis


### Guardar información 

In [6]:
#Guardar en csv

df_proyectos.to_csv("conflictos_ambientales_raw.csv", index=False, encoding='utf-8')

print("[OK] El archivo 'conflictos_ambientales_raw.csv' se generó correctamente.")

[OK] El archivo 'conflictos_ambientales_raw.csv' se generó correctamente.
